# Musanga V4 — NB02: RGB-only U-Net, spatial CV on 78 field plots

This is the **first trainable V4 model**, not a multimodal ablation. Read the **FINAL** NB01 manifest and frozen overlap-group folds. Learn from exhaustive field plots, KVO-corrected exhaustive circles outside plots, and explicit partial crowns (unknown background ignored). Evaluate only held-out field plots with observed RGB data, reporting per-plot, per-site and macro results rather than relying on pooled Yangambi-dominated pixels.

**Recommended first run:** `SMOKE_TEST=True`, `RUN_FOLDS=[0]`; then switch to full spatial CV (`SMOKE_TEST=False`, `RUN_FOLDS=[0,1,2]`). This notebook does not alter V3/V3_1 or the NB00/NB01 reference files. Actual model weights/results are saved in `Outputs/v4/02_rgb_unet_field78/`.


In [ ]:
# 00 — Environment, frozen inputs and run settings
%pip -q install geopandas pyogrio shapely rasterio scipy scikit-learn torch matplotlib
from pathlib import Path
import json, math, random, copy, time, warnings, os
import numpy as np, pandas as pd, geopandas as gpd, rasterio
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from rasterio.windows import Window
from rasterio.features import rasterize
from shapely.geometry import box
from shapely.ops import unary_union
from shapely import make_valid
from shapely.strtree import STRtree
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
DRIVE=Path('/content/drive/MyDrive')
ROOT=DRIVE/'Colab Notebooks/Musanga/Final'
assert ROOT.is_dir(), ROOT
DATA=ROOT/'Data'
REF_DIR=ROOT/'Outputs/v4/01_reference_field78_combined'
CONTRACT=json.loads((REF_DIR/'01_field78_reference_contract.json').read_text())
assert CONTRACT['schema']=='musanga_v4_field78_combined_reference_final_2',CONTRACT['schema']
NB00=json.loads(Path(CONTRACT['input_manifest']).read_text())
assert NB00['n_field_plots']==78
SPLITS=pd.read_csv(CONTRACT['cv_manifest'])
assert len(SPLITS)==9096,(len(SPLITS),'Expected 3032 unique windows × 3 folds; changed source audit?')
K=['cv_fold','mosaic','row','col','width','height']
assert not SPLITS.duplicated(K).any()
assert SPLITS.groupby('cv_fold').size().eq(3032).all()
assert set(SPLITS.split).issubset({'train','val','excluded_spatial_guard','excluded_no_validation_pixels'})
PLOTS=gpd.read_file(NB00['field_support_qa']).to_crs('EPSG:32635')
PLOT_FOLDS=pd.read_csv(CONTRACT['folds_csv'])
assert len(PLOTS)==len(PLOT_FOLDS)==78 and PLOT_FOLDS.source_id.is_unique
PLOTS=PLOTS.merge(PLOT_FOLDS[['source_id','site_key','cv_fold']],left_on='Plot_name',right_on='source_id',validate='one_to_one')
CIRCLE_FOLDS=pd.read_csv(REF_DIR/'01_field78_circle_folds.csv')
# Adjustable, deterministic experiment settings
SMOKE_TEST=True            # FIRST RUN; switch False for real training
RUN_FOLDS=[0]              # change to [0,1,2] for final evaluation
SEED=20261113
CHIP=256                   # 256 native pixels; no spatial downsampling
BATCH_SIZE=4
EPOCHS=2 if SMOKE_TEST else 20
STEPS_PER_EPOCH=30 if SMOKE_TEST else 250
LEARNING_RATE=2e-4
PATIENCE=4
POSITIVE_SAMPLE_BOOST=2.5
MAX_VAL_WINDOWS_PER_SITE=8 if SMOKE_TEST else None
NUM_WORKERS=0              # Google Drive + GDAL: safest in Colab
OUT=ROOT/'Outputs/v4/02_rgb_unet_field78'/('smoke' if SMOKE_TEST else 'full_spatial_cv')
OUT.mkdir(parents=True,exist_ok=True)
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',DEVICE,'| train mode:', 'SMOKE TEST' if SMOKE_TEST else 'FULL SPATIAL CV')
print('Final manifest:',CONTRACT['cv_manifest'])
print('Observed window splits:');display(SPLITS.groupby(['cv_fold','site','split']).size().unstack(fill_value=0))


## 01 — Load and repair vector supervision once; verify exhaustive supports

The 78 plots and 34 circles are separate exhaustive areas. Any circle area inside a field plot is overridden by its field-plot annotation. Outside exhaustive areas, partial-site polygons label **their own pixels only**. Geometries are repaired conservatively (≤5% area shift), preserving source files. An explicit negative pixel is not inferred outside plots/circles.


In [ ]:
# 01 — Geometry references and fast spatial indexes
CROWNS=gpd.read_file(NB00['exhaustive_plot_crowns']).to_crs('EPSG:32635')
CIRCLE_ROI=gpd.read_file(NB00['circle_complete_roi']).to_crs('EPSG:32635')
CIRCLE_POS=gpd.read_file(NB00['circle_positive_crowns']).to_crs('EPSG:32635')
assert len(CROWNS)==3324 and len(CIRCLE_ROI)==34
assert 'Class_ID' in CROWNS
def polygons(g):
    if g is None or g.is_empty:return []
    if g.geom_type=='Polygon':return [g]
    if g.geom_type=='MultiPolygon':return list(g.geoms)
    return [a for v in getattr(g,'geoms',[]) for a in polygons(v)]
def fix(g):
    if g is None or g.is_empty:return None
    if g.is_valid:return g
    q=unary_union(polygons(make_valid(g)))
    if q.is_empty or not q.is_valid:return None
    if abs(q.area-g.area)/max(q.area,g.area,1e-12)>.05:return None
    return q
CROWNS.geometry=[fix(g) for g in CROWNS.geometry]
assert CROWNS.geometry.notna().all(),'Unrepaired plot crowns; see NB00 QA'
CIRCLE_ROI.geometry=[fix(g) for g in CIRCLE_ROI.geometry]
CIRCLE_POS.geometry=[fix(g) for g in CIRCLE_POS.geometry]
assert CIRCLE_ROI.geometry.notna().all()
CIRCLE_ROI['circle_id']=np.arange(len(CIRCLE_ROI))
CIRCLE_ROI=CIRCLE_ROI.merge(CIRCLE_FOLDS[['circle_id','site_key','cv_fold',
    'cross_fold_plot_intersection']],on='circle_id',validate='one_to_one')
CROWN_POS=list(CROWNS.loc[pd.to_numeric(CROWNS.Class_ID).eq(1)].geometry)
CROWN_NEG=list(CROWNS.loc[~pd.to_numeric(CROWNS.Class_ID).eq(1)].geometry)
CIRCLE_POS_GEOMS=[g for g in CIRCLE_POS.geometry if g is not None and not g.is_empty]
# Partial site references from old V3_1 audit, same positive=1 / other known class negative.
PARTIAL={}
for site,path in NB00['partial_site_crowns'].items():
    d=gpd.read_file(path).to_crs('EPSG:32635')
    d.geometry=[fix(g) for g in d.geometry]
    d=d[d.geometry.notna() & pd.to_numeric(d.Class_ID,errors='coerce').notna()]
    PARTIAL[site]=d
P_GEOMS=list(PLOTS.geometry)
C_GEOMS=list(CIRCLE_ROI.geometry)
P_T=STRtree(P_GEOMS); C_T=STRtree(C_GEOMS)
CP_T=STRtree(CROWN_POS);CN_T=STRtree(CROWN_NEG)
KPOS_T=STRtree(CIRCLE_POS_GEOMS)
PARTIAL_T={}
for site,d in PARTIAL.items():
    pos=list(d.loc[pd.to_numeric(d.Class_ID).eq(1)].geometry)
    neg=list(d.loc[~pd.to_numeric(d.Class_ID).eq(1)].geometry)
    PARTIAL_T[site]=(pos,neg,STRtree(pos) if pos else None,
                     STRtree(neg) if neg else None)
RGB={}
for item in sorted(SPLITS.mosaic.unique()):
    q=DATA/'Ortho_RGB'/item
    assert q.exists(),q
    with rasterio.open(q) as ds:
        assert ds.crs.to_epsg()==32635,(item,ds.crs)
        assert ds.count>=3,(item,ds.count)
        RGB[item]=dict(path=str(q),transform=ds.transform,site=next(
            iter(SPLITS.loc[SPLITS.mosaic==item,'site'].unique())))
# NOTE: ROI.shp is the authoritative six-mosaic modelling ROI (not the old
# circle-only support). The validation/labelled field plot is also required.
ROIS=gpd.read_file(DATA/'Ortho_RGB/ROI.shp').to_crs('EPSG:32635')
ROI_GEOMS=list(ROIS.geometry)
ROI_T=STRtree(ROI_GEOMS)
print('Loaded',len(PLOTS),'plots,',len(CIRCLE_ROI),'circles,',
      len(CROWN_POS),'Musanga plot crowns and',len(CIRCLE_POS_GEOMS),'circle crowns')


In [ ]:
# 02 — Rasterize strict plot > circle > partial supervision on demand
# Geometry-level spatial queries restrict operations to the local 256px footprint.
def geom_hits(tree,geoms,ext):
    if tree is None:return []
    return [geoms[int(i)] for i in tree.query(ext,predicate='intersects')]
def rgb_window(raster_path,win):
    with rasterio.open(raster_path) as ds:
        a=ds.read([1,2,3],window=win,masked=True)
        valid=ds.dataset_mask(window=win)>0
        valid &= ~np.ma.getmaskarray(a).any(axis=0)
        x=np.asarray(a.filled(0),dtype=np.float32)
        # RGB products expected standard 8-bit; prevent silent incorrect scaling.
        if np.issubdtype(np.dtype(ds.dtypes[0]),np.integer) and np.dtype(ds.dtypes[0]).itemsize==1:
            x=x/255.
        else:
            raise RuntimeError(f'Non-uint8 RGB {ds.dtypes}: {raster_path}; define radiometric scaling first')
    return x,np.asarray(valid,bool)
def supervision(mosaic,win,fold,role):
    site=RGB[mosaic]['site'];t=RGB[mosaic]['transform']
    h,w=int(win.height),int(win.width)
    affine=rasterio.windows.transform(win,t)
    footprint=box(*rasterio.windows.bounds(win,t))
    def burn(gs):
        return rasterize(((g,1) for g in gs if g is not None and not g.is_empty),
            out_shape=(h,w),transform=affine,fill=0,dtype='uint8').astype(bool) if gs else np.zeros((h,w),bool)
    roi=burn(geom_hits(ROI_T,ROI_GEOMS,footprint))
    pids=[int(i) for i in P_T.query(footprint,predicate='intersects')
          if PLOTS.iloc[int(i)].site_key==site and
          (role=='train' and PLOTS.iloc[int(i)].cv_fold!=fold or
           role=='val' and PLOTS.iloc[int(i)].cv_fold==fold)]
    plot=burn([P_GEOMS[i] for i in pids])
    # Every other fold's field plot is out of scope for val; all exhaustive
    # plot geometries (not only selected) override circle labels.
    allplot=burn([P_GEOMS[int(i)] for i in P_T.query(footprint,predicate='intersects')
                  if PLOTS.iloc[int(i)].site_key==site])
    cids=[int(i) for i in C_T.query(footprint,predicate='intersects')
          if CIRCLE_ROI.iloc[int(i)].site_key==site and
          not CIRCLE_ROI.iloc[int(i)].cross_fold_plot_intersection and
          (role=='train' and CIRCLE_ROI.iloc[int(i)].cv_fold!=fold)]
    circle=burn([C_GEOMS[i] for i in cids])&~allplot if role=='train' else np.zeros((h,w),bool)
    posplot=burn(geom_hits(CP_T,CROWN_POS,footprint))&plot
    negplot=burn(geom_hits(CN_T,CROWN_NEG,footprint))&plot
    poscircle=burn(geom_hits(KPOS_T,CIRCLE_POS_GEOMS,footprint))&circle
    part_pos=np.zeros((h,w),bool);part_neg=np.zeros((h,w),bool)
    if role=='train':
        positives,negatives,tp,tn=PARTIAL_T[site]
        part_pos=burn(geom_hits(tp,positives,footprint))&~allplot&~circle
        part_neg=burn(geom_hits(tn,negatives,footprint))&~allplot&~circle
    target=(posplot|poscircle|part_pos)
    conflicting=(posplot&negplot)|(part_pos&part_neg)
    weight=roi&(plot|circle|part_pos|part_neg)&~conflicting
    return target.astype(np.float32),weight.astype(np.float32),plot.astype(bool)
def get_chip(mosaic,win,fold,role):
    x,rgbvalid=rgb_window(RGB[mosaic]['path'],win)
    y,w,plot=supervision(mosaic,win,fold,role)
    w*=rgbvalid.astype(np.float32)
    return x,y,w,plot
print('Prepared lazy RGB + spatially strict masks. No imagery loaded globally.')


## 02 — Preflight QA and site-balanced training sampling

The preflight samples held-out plots and training locations with the **real RGB validity mask**, not just geometric ROI. The training sampler gives comparable chances to Baego, MOS, Yangambi and Yoko, and boosts windows likely to contain positive reference crowns. Spatial folds and real validation prevalence stay unchanged. The 10 ha ForestGEO area is **not discarded**, but each plot contributes equally to model selection and final macro metrics.


In [ ]:
# 03 — Preflight and training sample weights, without reading full mosaics
def estimate_pos_window(mosaic,row,col,width,height):
    ext=box(*rasterio.windows.bounds(Window(col,row,width,height),RGB[mosaic]['transform']))
    return bool(len(CP_T.query(ext,predicate='intersects')) or
                len(KPOS_T.query(ext,predicate='intersects')))
eligible=SPLITS[SPLITS.split.isin(['train','val'])].copy()
eligible['likely_positive']=False
for idx,r in eligible.iterrows():
    eligible.at[idx,'likely_positive']=estimate_pos_window(
        r.mosaic,int(r.row),int(r.col),int(r.width),int(r.height))
print('Window / likely-positive counts (not verified prevalence):')
display(eligible.groupby(['cv_fold','site','split']).agg(
    windows=('mosaic','size'),likely_positive=('likely_positive','sum')))
# Test a small image sample from every mosaic; fail clearly on unreadable/invalid RGB.
for mosaic in sorted(RGB):
    rows=eligible[eligible.mosaic.eq(mosaic)]
    assert len(rows),(mosaic,'has no candidate windows')
    r=rows.iloc[0];win=Window(int(r.col),int(r.row),
                              min(256,int(r.width)),min(256,int(r.height)))
    x,mask=rgb_window(RGB[mosaic]['path'],win)
    print(mosaic,'RGB validity sample:',round(float(mask.mean()),3),'range:',x.min(),x.max())
assert DEVICE.type=='cuda' or SMOKE_TEST,'Select a GPU runtime for full U-Net training'


In [ ]:
# 04 — Lazy training and held-out validation datasets; no full mosaics cached
class Windows(Dataset):
    def __init__(self,frame,fold,role,chip=CHIP):
        assert role in ('train','val')
        self.role=role;self.fold=fold;self.chip=chip
        if role=='val':
            # Disjoint subwindows cover all original validation windows.
            tiles=[]
            for r in frame.itertuples(index=False):
                for dy in range(0,int(r.height),chip):
                    for dx in range(0,int(r.width),chip):
                        hh=min(chip,int(r.height)-dy)
                        ww=min(chip,int(r.width)-dx)
                        if hh==chip and ww==chip:
                            tiles.append((r.mosaic,int(r.row)+dy,int(r.col)+dx,
                                          r.site))
            self.items=tiles
        else:
            self.items=[(r.mosaic,int(r.row),int(r.col),int(r.width),
                         int(r.height),r.site) for r in frame.itertuples(index=False)]
    def __len__(self):return len(self.items)
    def __getitem__(self,i):
        if self.role=='train':
            mosaic,row,col,w,h,site=self.items[i]
            # Only crop *within* the candidate window.
            rr=row+random.randint(0,max(0,h-self.chip))
            cc=col+random.randint(0,max(0,w-self.chip))
            ww=min(w,self.chip);hh=min(h,self.chip)
        else:
            mosaic,rr,cc,site=self.items[i]
            ww=hh=self.chip
        win=Window(cc,rr,ww,hh)
        x,y,weight,_=get_chip(mosaic,win,self.fold,self.role)
        # Right/bottom edge patches are excluded for val; pad train if needed.
        if hh<self.chip or ww<self.chip:
            pad=((0,0),(0,self.chip-hh),(0,self.chip-ww))
            x=np.pad(x,pad)
            y=np.pad(y,((0,self.chip-hh),(0,self.chip-ww)))
            weight=np.pad(weight,((0,self.chip-hh),(0,self.chip-ww)))
        return torch.from_numpy(x),torch.from_numpy(y[None]),torch.from_numpy(weight[None]),(
            mosaic,int(rr),int(cc),site)
def make_train_sampler(frame):
    # Balanced site *and* approximate positive support. Every candidate has nonzero probability.
    counts=frame.site.value_counts().to_dict()
    boosts=np.where(frame.likely_positive.to_numpy(bool),POSITIVE_SAMPLE_BOOST,1.)
    weights=boosts/np.array([counts[s] for s in frame.site],float)
    return WeightedRandomSampler(torch.as_tensor(weights,dtype=torch.double),
                num_samples=STEPS_PER_EPOCH*BATCH_SIZE,replacement=True)


## 03 — Train RGB U-Net and preserve the best checkpoint per spatial fold

The validation loss is calculated only for labelled **held-out field-plot pixels with valid RGB**. A weighted BCE + soft Dice objective supports scarce positive pixels. The smoke test runs two short epochs on fold 0; it is **not** scientific performance evidence. For full CV, run three folds and keep every plot's final confusion statistics.


In [ ]:
# 05 — Compact 4-level U-Net with masked BCE + Dice
class Block(nn.Module):
    def __init__(self,c_in,c_out):
        super().__init__()
        self.net=nn.Sequential(nn.Conv2d(c_in,c_out,3,padding=1,bias=False),
            nn.BatchNorm2d(c_out),nn.ReLU(inplace=True),
            nn.Conv2d(c_out,c_out,3,padding=1,bias=False),
            nn.BatchNorm2d(c_out),nn.ReLU(inplace=True))
    def forward(self,x):return self.net(x)
class UNet(nn.Module):
    def __init__(self,base=24):
        super().__init__()
        self.b1=Block(3,base);self.b2=Block(base,base*2)
        self.b3=Block(base*2,base*4);self.mid=Block(base*4,base*8)
        self.pool=nn.MaxPool2d(2)
        self.d3=Block(base*12,base*4);self.d2=Block(base*6,base*2)
        self.d1=Block(base*3,base);self.head=nn.Conv2d(base,1,1)
    def forward(self,x):
        a=self.b1(x);b=self.b2(self.pool(a));c=self.b3(self.pool(b))
        d=self.mid(self.pool(c))
        x=self.d3(torch.cat([nn.functional.interpolate(d,size=c.shape[-2:],mode='bilinear',align_corners=False),c],1))
        x=self.d2(torch.cat([nn.functional.interpolate(x,size=b.shape[-2:],mode='bilinear',align_corners=False),b],1))
        x=self.d1(torch.cat([nn.functional.interpolate(x,size=a.shape[-2:],mode='bilinear',align_corners=False),a],1))
        return self.head(x)
def masked_loss(logits,y,weight):
    if weight.sum()<=0:return logits.sum()*0
    bce=nn.functional.binary_cross_entropy_with_logits(logits,y,reduction='none')
    bce=(bce*weight).sum()/weight.sum()
    p=torch.sigmoid(logits)
    intersection=(p*y*weight).sum()
    dice=1-(2*intersection+1.)/((p*weight).sum()+(y*weight).sum()+1.)
    return .7*bce+.3*dice
def run_epoch(model,loader,optimizer=None):
    model.train(optimizer is not None)
    tot=0.;npix=0
    for x,y,w,meta in loader:
        x=x.to(DEVICE);y=y.to(DEVICE);w=w.to(DEVICE)
        if w.sum().item()==0:continue
        with torch.set_grad_enabled(optimizer is not None):
            logits=model(x);loss=masked_loss(logits,y,w)
            if optimizer is not None:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                optimizer.step()
        n=w.sum().item()
        tot+=loss.item()*n;npix+=n
    return tot/max(npix,1)


In [ ]:
# 06 — Run selected folds with site-balanced training and early stopping
HISTORY=[]
CHECKPOINTS={}
for fold in RUN_FOLDS:
    assert fold in (0,1,2)
    tr=eligible[(eligible.cv_fold==fold)&(eligible.split=='train')].copy()
    va=eligible[(eligible.cv_fold==fold)&(eligible.split=='val')].copy()
    if SMOKE_TEST:
        va=va.groupby('site',group_keys=False).head(MAX_VAL_WINDOWS_PER_SITE)
    assert len(tr)>0 and len(va)>0
    train_loader=DataLoader(Windows(tr,fold,'train'),batch_size=BATCH_SIZE,
         sampler=make_train_sampler(tr),num_workers=NUM_WORKERS,pin_memory=DEVICE.type=='cuda')
    val_ds=Windows(va,fold,'val')
    val_loader=DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,
        num_workers=NUM_WORKERS,pin_memory=DEVICE.type=='cuda')
    model=UNet().to(DEVICE)
    optim=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=1e-4)
    best=float('inf');wait=0
    ckpt=OUT/f'rgb_unet_fold{fold}_best.pt'
    for epoch in range(1,EPOCHS+1):
        tic=time.time()
        trainloss=run_epoch(model,train_loader,optim)
        valloss=run_epoch(model,val_loader,None)
        row=dict(fold=fold,epoch=epoch,train_loss=trainloss,val_loss=valloss,
                 elapsed_s=round(time.time()-tic,1))
        HISTORY.append(row)
        print(row)
        if valloss<best-1e-4:
            best=valloss;wait=0
            torch.save(dict(model=model.state_dict(),fold=fold,
                config=dict(chip=CHIP,base=24,seed=SEED,smoke=SMOKE_TEST),
                epoch=epoch,val_loss=best),ckpt)
        else:
            wait+=1
            if wait>=PATIENCE:break
    CHECKPOINTS[fold]=ckpt
pd.DataFrame(HISTORY).to_csv(OUT/'02_rgb_training_history.csv',index=False)
print('Best checkpoints:',CHECKPOINTS)


## 04 — Held-out accuracy by **individual plot** (not pooled pixels)

Evaluate the best checkpoint at a fixed preliminary threshold of 0.5. Each prediction contributes to a **single held-out plot**, with a special mask for that plot; only valid RGB and ROI pixels are counted. Report per-plot TP/FP/FN/TN, cover bias, MAE, IoU, Dice/F1, precision and recall. A zero-positive plot has undefined recall/F1, and is scored for false positives and cover error. Summaries use equal plot weights, and separately equal site weights. Threshold optimization, if eventually used, must be **nested within training data**, not fitted on the held-out plots.


In [ ]:
# 07 — Plot-wise confusion, fold-/site-balanced summaries and output provenance
def burn_single(g,win,affine):
    return rasterize([(g,1)],out_shape=(int(win.height),int(win.width)),
        transform=affine,fill=0,dtype='uint8').astype(bool)
PLOT_ROWS=[]
for fold in RUN_FOLDS:
    ck=torch.load(CHECKPOINTS[fold],map_location=DEVICE,weights_only=False)
    model=UNet().to(DEVICE);model.load_state_dict(ck['model']);model.eval()
    fval=SPLITS[(SPLITS.cv_fold==fold)&(SPLITS.split=='val')].copy()
    if SMOKE_TEST:
        fval=fval.groupby('site',group_keys=False).head(MAX_VAL_WINDOWS_PER_SITE)
    plot_state={}
    psite=PLOTS[PLOTS.cv_fold==fold]
    for r in psite.itertuples(index=False):
        plot_state[str(r.Plot_name)]=dict(tp=0,fp=0,fn=0,tn=0,n=0,
                                          site=r.site_key,ba=float(r.pct_BA_musanga))
    with torch.no_grad():
        for row in fval.itertuples(index=False):
            # Disjoint tiles from 512px windows; no duplicate physical pixels.
            for dr in range(0,int(row.height),CHIP):
                for dc in range(0,int(row.width),CHIP):
                    hh=min(CHIP,int(row.height)-dr);ww=min(CHIP,int(row.width)-dc)
                    if hh!=CHIP or ww!=CHIP:continue
                    win=Window(int(row.col)+dc,int(row.row)+dr,CHIP,CHIP)
                    _,weights,_=supervision(row.mosaic,win,fold,'val')
                    x,valid=rgb_window(RGB[row.mosaic]['path'],win)
                    if not np.any(valid&weights.astype(bool)):continue
                    prediction=torch.sigmoid(model(torch.from_numpy(x[None]).to(DEVICE)))[0,0]
                    pred=prediction.cpu().numpy()>=.5
                    # Compute reference once. Plot identity comes from *actual*
                    # field geometry, not from a candidate window's source_id.
                    truth,base_weight,_=supervision(row.mosaic,win,fold,'val')
                    affine=rasterio.windows.transform(win,RGB[row.mosaic]['transform'])
                    ext=box(*rasterio.windows.bounds(win,RGB[row.mosaic]['transform']))
                    candidates=P_T.query(ext,predicate='intersects')
                    for i in candidates:
                        p=PLOTS.iloc[int(i)]
                        if p.cv_fold!=fold or p.site_key!=row.site:continue
                        plotmask=burn_single(p.geometry,win,affine)
                        mask=plotmask&valid&base_weight.astype(bool)
                        if not mask.any():continue
                        obs=truth.astype(bool)
                        entry=plot_state[str(p.Plot_name)]
                        entry['tp']+=int((mask&pred&obs).sum())
                        entry['fp']+=int((mask&pred&~obs).sum())
                        entry['fn']+=int((mask&~pred&obs).sum())
                        entry['tn']+=int((mask&~pred&~obs).sum())
                        entry['n']+=int(mask.sum())
    for name,d in plot_state.items():
        tp,fp,fn,tn=d['tp'],d['fp'],d['fn'],d['tn']
        n=d['n'];pos=tp+fn;predpos=tp+fp
        PLOT_ROWS.append(dict(fold=fold,Plot_name=name,site=d['site'],
             n_rgb_valid_label_pixels=n,field_BA_pct=d['ba'],
             tp=tp,fp=fp,fn=fn,tn=tn,
             crown_reference_pct=100*pos/n if n else np.nan,
             predicted_cover_pct=100*predpos/n if n else np.nan,
             cover_bias_pp=100*(predpos-pos)/n if n else np.nan,
             cover_abs_error_pp=100*abs(predpos-pos)/n if n else np.nan,
             pixel_precision=tp/(tp+fp) if tp+fp else np.nan,
             pixel_recall=tp/pos if pos else np.nan,
             pixel_F1=2*tp/(2*tp+fp+fn) if pos and (2*tp+fp+fn)>0 else np.nan,
             pixel_IoU=tp/(tp+fp+fn) if pos and tp+fp+fn>0 else np.nan,
             false_positive_fraction_pct=100*fp/n if n else np.nan))
RESULTS=pd.DataFrame(PLOT_ROWS)
RESULTS.to_csv(OUT/'02_rgb_heldout_per_plot.csv',index=False)
SUMMARY=RESULTS.groupby(['fold','site'],as_index=False).agg(
    n_plots=('Plot_name','size'),
    plots_with_evaluation=('n_rgb_valid_label_pixels',lambda x:int((x>0).sum())),
    mean_cover_MAE_pp=('cover_abs_error_pp','mean'),
    mean_pixel_IoU=('pixel_IoU','mean'),
    mean_pixel_F1=('pixel_F1','mean'),
    mean_false_positive_fraction_pct=('false_positive_fraction_pct','mean'))
SUMMARY.to_csv(OUT/'02_rgb_heldout_by_fold_site.csv',index=False)
display(SUMMARY.round(3))
print('Equal-site mean plot MAE (pp):',
      round(float(SUMMARY.mean_cover_MAE_pp.mean()),3))
if SMOKE_TEST:
    print('SMOKE TEST ONLY: fold 0, partial validation windows; do not use as scientific accuracy.')
else:
    assert set(RESULTS.fold)=={0,1,2}
    print('Full 3-fold spatial CV results saved.')
RUN_INFO=dict(schema='musanga_v4_rgb_unet_baseline_1',
    smoke_test=SMOKE_TEST,folds=RUN_FOLDS,epochs=EPOCHS,
    patches_per_epoch=STEPS_PER_EPOCH*BATCH_SIZE,chip=CHIP,
    final_manifest=CONTRACT['cv_manifest'],threshold=.5,
    macro_rule='per plot, then per site; zero-positive F1 undefined',
    checkpoints={str(f):str(p) for f,p in CHECKPOINTS.items()},
    model_status='trained_smoke_only' if SMOKE_TEST else 'trained_full_spatial_cv')
(OUT/'02_run_contract.json').write_text(json.dumps(RUN_INFO,indent=2))
